# MARV × Titans — does the forgetting curve hold with REAL text? (Colab T4)

Every experiment on `marv-titan` so far (`titans_memdiff.py`, `titans_ablation.py`,
`titans_per_unit.py`) fed the memory random 64-dim vectors. That's deliberate — it isolates
the memory mechanism from language modeling — but it leaves open whether the same forgetting
curve and diffuse-storage story hold once the memory is reading actual language instead of
noise, and whether real language creates any structure random vectors can't.

This notebook trains a small, real, byte-level language model with a Titans memory wired in
(`titans_pytorch.MemoryAsContextTransformer`, the "MAC" architecture) on enwik8 (Wikipedia
text), then re-runs the same early-vs-end weight-snapshot diff on the memory as it reads a real
held-out passage.

**Scaled down from the library's own `train_mac.py` recipe on purpose** (that one is dim=384,
depth=8, 100k batches, wandb, flex-attention — a real multi-hour+ run, not a Colab demo). Here:
dim=64 (matches the memory's own `dim_head`, so results are directly comparable to the
random-vector notebooks), depth=4, one memory layer, no flex-attention. This will NOT produce
a good language model — it's a correctness + qualitative-structure check, not a real LM.

In [ ]:
!pip install -q titans-pytorch
!git clone -q -b marv-titan https://github.com/thebnbrkr/marv.git /content/marv
!git clone -q --depth 1 https://github.com/lucidrains/titans-pytorch.git /content/titans-pytorch-src
import sys; sys.path.insert(0, '/content/marv/experiments')

import os
# the enwik8 dataset ships inside the titans-pytorch repo itself (data/enwik8.gz) --
# if this ever moves, download it directly from http://prize.hutter1.net/ instead.
DATA_PATH = '/content/titans-pytorch-src/data/enwik8.gz'
assert os.path.exists(DATA_PATH), 'enwik8.gz not found -- see the comment above for a fallback source'

import time, numpy as np, torch, matplotlib.pyplot as plt
from titans_real_text import (
    build_model, load_enwik8, sample_batch, train,
    diff_memory_on_passage, inspect_decay_gate,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
print('device:', device)

## 1. Load enwik8, build the small MAC transformer

`neural_memory_model=MemoryMLP(64, depth=2, expansion_factor=4.)` gives the memory the exact
same 64→256→64 shape used in every prior notebook on this branch — so the numbers below are
directly comparable to the random-vector experiments, not just qualitatively similar.

In [ ]:
data_train, data_val = load_enwik8(DATA_PATH)
print(f'train bytes: {len(data_train):,}   val bytes: {len(data_val):,}')

model = build_model().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'model: {n_params/1e6:.2f}M params')

## 2. Train

On a T4 this should run noticeably faster than the ~1.3s/step seen on CPU locally — feel free
to raise `STEPS` well past the default if you want a stronger (if still small) model. The loss
will not get near the library's own reported numbers at this scale; watch for it dropping
below the byte-uniform baseline (`ln(256) ≈ 5.545` nats) and continuing to fall, which is the
actual thing being checked here.

In [ ]:
STEPS = 3000
SEQ_LEN = 256
BATCH_SIZE = 16

t0 = time.time()
train(model, data_train, data_val, steps=STEPS, seq_len=SEQ_LEN, batch_size=BATCH_SIZE, lr=2e-4, device=device)
print(f'\n{STEPS} steps took {time.time()-t0:.0f}s')

## 3. Diff the memory on a real held-out passage

Same early-vs-end snapshot diff as `titans_memdiff.py` (`gate_cos`, `norm_ratio`, direction/
magnitude retained on the hardest-written units) — except the "document" is now real English
text from held-out enwik8, not 96 random vectors.

In [ ]:
passage = sample_batch(data_val, 512, 1)[0]
print('passage (decoded):')
print(repr(bytes(passage[:200].tolist()).decode('utf-8', errors='replace')))
print()
diff_memory_on_passage(model, passage, device)

## 4. WHY does it forget less on real text? Check the gate directly

Everything above shows the memory *behaves* as if it forgets less on real text than on the
toy recall task. That leaves an open question: is the forget gate actually **reading the
input** and choosing to retain real-looking text, or did training just settle on a fixed,
input-independent low-forgetting habit that would apply to anything, gibberish included?

This hooks the memory layer's own `to_decay_factor` module (the thing that produces α_t) during
a real forward pass on the SAME real passage above, then again on random bytes through the
exact same trained weights, and compares the two gate values directly.

In [ ]:
inspect_decay_gate(model, passage, device)

**How to read it:** if the gate value is noticeably *lower* on real text than on random bytes,
that's direct evidence the gate is genuinely input-sensitive — it's making a live decision to
retain predictable, real-looking content specifically. If the two are close, the low forgetting
we measured is a fixed habit the training run settled into, not something reacting to what it's
currently reading — a different, less exciting mechanism than "the network is smart about what
to keep," but an important distinction either way. On a lightly-trained (30-step) local model
this came out ~equal (gap +0.002) — but that model hadn't learned much of anything yet. The real
answer needs your properly-trained model here.

## 5. Save the checkpoint and upload to Hugging Face

This is a **custom architecture** (`titans_pytorch.MemoryAsContextTransformer`), not a standard
`transformers` model — there's no `model.push_to_hub()`. We save the raw weights, create a repo,
upload the file plus a model card explaining how to reload it (needs `titans_pytorch` and this
branch's `build_model()` function).

Needs a Hugging Face token with write access (from https://huggingface.co/settings/tokens) —
`notebook_login()` below prompts for it securely inside Colab; nothing gets typed into a cell.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
HF_REPO_ID = 'your-username/titans-marv-enwik8'  # <-- change this to yours

import torch
CKPT_PATH = 'titans_marv_enwik8.pt'
torch.save(model.state_dict(), CKPT_PATH)
print(f'saved {CKPT_PATH}')

In [ ]:
with torch.no_grad():
    model.eval()
    _final_val_batch = sample_batch(data_val, SEQ_LEN, BATCH_SIZE).to(device)
    final_val_loss = model(_final_val_batch, return_loss=True).item()

MODEL_CARD = f"""---
license: mit
tags:
- titans
- test-time-memory
- marv
---

# Titans MAC transformer (MARV research checkpoint)

A small byte-level language model with a Titans neural memory ({n_params/1e6:.2f}M params),
trained on enwik8 as part of research into whether test-time memory (weights that update via
gradient descent *during inference*) can be feature-diffed and causally ablated the way MARV
already does for frozen model weights.

Trained {STEPS} steps, final val loss {final_val_loss:.3f} nats (byte-uniform baseline: 5.545).

See the `marv-titan` branch of https://github.com/thebnbrkr/marv for the full writeup
(`experiments/README.md`) and the code needed to reload this checkpoint:

```python
# needs: pip install titans-pytorch, and titans_real_text.py from the marv-titan branch
from titans_real_text import build_model
import torch

model = build_model()
model.load_state_dict(torch.load("titans_marv_enwik8.pt", map_location="cpu"))
model.eval()
```

Findings from this checkpoint's analysis: the memory, trained on real language modeling,
accumulates writes rather than forgetting them (norm_ratio > 1, magnitude never shrinks below
its original size) -- the OPPOSITE of the aggressive exponential forgetting seen when the same
architecture is trained on a toy autoassociative-recall task instead. See the repo for details.
"""

with open("README.md", "w") as f:
    f.write(MODEL_CARD)
print("wrote README.md")
print(f"final val loss used in model card: {final_val_loss:.3f}")

In [ ]:
from huggingface_hub import create_repo, upload_file

create_repo(HF_REPO_ID, private=False, exist_ok=True)  # private=True instead if you'd rather not make it public

upload_file(path_or_fileobj=CKPT_PATH, path_in_repo=CKPT_PATH, repo_id=HF_REPO_ID)
upload_file(path_or_fileobj='README.md', path_in_repo='README.md', repo_id=HF_REPO_ID)

print(f'uploaded to https://huggingface.co/{HF_REPO_ID}')

## What to look for

- **Does `norm_ratio` on the moved units still show a clear pattern** (writes decaying or
  accumulating), the way the random-vector version did? Real text is far from IID random
  vectors — repeated bytes, common words, and predictable structure could make the memory's
  write/forget behavior look different (e.g. less each token is genuinely "surprising").
- **Does the direction-retained / magnitude-retained pattern for the hardest-written units**
  look like the same exponential-ish forgetting curve, or does real text's redundancy change
  the shape?
- This is still a small, briefly-trained model — a difference here could mean "real text
  changes the story" or just "this model hasn't trained long enough to show it." Worth
  comparing at a few different `STEPS` values before drawing a firm conclusion.

**Next steps (not done here):** a `describe_feature`-style logit lens reading what a specific
hidden unit promotes (e.g. "unit 33 now writes toward the letter e"), and re-running the
`titans_per_unit.py` per-unit-decay localization test with this real-text-trained memory
instead of random vectors. See `experiments/README.md` roadmap item 3 on the `marv-titan`
branch.